# PABPC1–RNA reference preparation — research prototype

**This is not a validated production protocol and it is not a 61-variant pipeline.** It prepares one reference candidate, records diagnostics, and stops if the chosen checks fail or the requested minimizer thresholds are not observed. It does not model longer RNAs, chemical modifications, or MM/GBSA. Do not use its candidate PDB for affinity scoring without further structural and convergence review.

The uploaded model has very serious overlaps. This prototype uses a **nonphysical soft-repulsion ramp** and a **stereochemistry-protecting penalty** before/during restrained OpenMM minimization. These alter the preparation pathway. The code requires the final penalty of each physical stage to be zero but that does not remove pathway bias or prove the biological pose is correct.

Input: your `revised_template.pdb` (or another name chosen at upload). The original is kept unchanged. Outputs: staged coordinates, state files, diagnostics, and—only if all programmed gates pass—a candidate solute PDB. Completed stages and saved unfinished-stage states can be resumed after an interruption if the input, code, seed, and OpenMM version match. Work inside a stage since its last saved state may need to be repeated. Default settings can take a long time, especially if Colab assigns no working GPU.

## 1. Install the pinned numerical environment

Tested locally with OpenMM 8.6.1 on CPU. GPU execution, Colab installation and full convergence have not been independently validated for every Colab runtime.

In [ ]:
import sys, subprocess
subprocess.run([sys.executable,'-m','pip','install','-q','openmm==8.6.1','numpy==2.4.6','scipy==1.17.1','biopython==1.88','openpyxl==3.1.5'],check=True)


## 2. Unpack the source and small component tests
The source is embedded here; the tests do not validate the experimental model.

In [ ]:
from pathlib import Path
import sys
TOOLS = Path('/content/pabpc1_reference_tools')
(TOOLS/'tests').mkdir(parents=True,exist_ok=True)
(TOOLS/'preflight.py').write_text('"""Conservative structural audit and input checks; not a minimization pipeline."""\nimport csv,json,math,hashlib,re\nfrom collections import Counter\nfrom pathlib import Path\nimport numpy as np\nfrom scipy.spatial import cKDTree\nfrom Bio.PDB import PDBParser\nfrom Bio.SeqUtils import seq1\n\nK_RESTRAINT=4184.0 # kJ mol^-1 nm^-2 for 0.5*k*r^2, 10 kcal mol^-1 A^-2\nHEADERS=[\'variant_id\',\'group\',\'rna_sequence\',\'modifications_json\',\'replicate_id\',\'notes\']\nRNA={\'A\',\'C\',\'G\',\'U\'}\nSUGAR={"O5\'","C5\'","C4\'","O4\'","C3\'","O3\'","C2\'","O2\'","C1\'"}\nBASE={\'A\':{\'N9\',\'C8\',\'N7\',\'C5\',\'C6\',\'N6\',\'N1\',\'C2\',\'N3\',\'C4\'}, \'G\':{\'N9\',\'C8\',\'N7\',\'C5\',\'C6\',\'O6\',\'N1\',\'C2\',\'N2\',\'N3\',\'C4\'},\'C\':{\'N1\',\'C2\',\'O2\',\'N3\',\'C4\',\'N4\',\'C5\',\'C6\'},\'U\':{\'N1\',\'C2\',\'O2\',\'N3\',\'C4\',\'O4\',\'C5\',\'C6\'}}\n\ndef sha256(p): return hashlib.sha256(Path(p).read_bytes()).hexdigest()\n\ndef validate_variants(path,expected_counts={\'benchmark\':34,\'control\':27}):\n with open(path,newline=\'\',encoding=\'utf-8-sig\') as f:\n  rd=csv.DictReader(f)\n  if rd.fieldnames!=HEADERS: raise ValueError(\'CSV headers must exactly equal \'+\',\'.join(HEADERS))\n  rows=list(rd)\n seen=set()\n for i,r in enumerate(rows,2):\n  if None in r or any(v is None for v in r.values()): raise ValueError(f\'Row {i}: incorrect number of fields\')\n  v=r[\'variant_id\'].strip(); group=r[\'group\']; seq=r[\'rna_sequence\'].strip().upper()\n  if not re.fullmatch(r\'[A-Za-z0-9_.-]+\',v) or v in (\'.\',\'..\') or v in seen: raise ValueError(f\'Row {i}: invalid or duplicate variant_id\')\n  seen.add(v)\n  if group not in (\'benchmark\',\'control\') or not r[\'replicate_id\'].strip(): raise ValueError(f\'Row {i}: invalid group or replicate ID\')\n  if len(seq)!=31 or set(seq)-RNA: raise ValueError(f\'Row {i}: sequence must contain exactly 31 A/C/G/U characters; variable-length structures require a separate validated builder\')\n  try: mods=json.loads(r[\'modifications_json\'])\n  except json.JSONDecodeError as e: raise ValueError(f\'Row {i}: malformed modifications JSON: {e}\') from e\n  if not isinstance(mods,list): raise ValueError(f\'Row {i}: modifications must be an array\')\n  used=set()\n  for m in mods:\n   if not isinstance(m,dict) or set(m)!={\'position\',\'modification\',\'parent_base\'}: raise ValueError(f\'Row {i}: each modification requires position, modification, parent_base\')\n   pos=m[\'position\']; parent=m[\'parent_base\']; mod=m[\'modification\']\n   if type(pos) is not int or not 1<=pos<=31 or pos in used or parent not in RNA or parent!=seq[pos-1] or not isinstance(mod,str) or not mod.strip(): raise ValueError(f\'Row {i}: invalid modification at {pos}\')\n   if mod==\'m6A\' and parent!=\'A\': raise ValueError(f\'Row {i}: m6A requires A\')\n   used.add(pos)\n  if (seq[16:]!=\'A\'*15 or any(p>=17 for p in used)) and \'[allow_A15_changes]\' not in r[\'notes\']: raise ValueError(f\'Row {i}: A15 changes require [allow_A15_changes] in notes\')\n  r[\'rna_sequence\']=seq; r[\'modification_status\']=\'requires_parameterization\' if mods else \'unmodified\'; r[\'parsed_modifications\']=mods\n if expected_counts is not None and Counter(r[\'group\'] for r in rows)!=Counter(expected_counts): raise ValueError(f\'Unexpected group counts: {dict(Counter(r["group"] for r in rows))}; expected {expected_counts}\')\n return rows\n\ndef audit_pdb(path,uniprot_sequence=None):\n path=Path(path); lines=path.read_text(errors=\'replace\').splitlines(); cryst=next((x for x in lines if x.startswith(\'CRYST1\')),None)\n box=None\n if cryst:\n  try: box=[float(cryst[6+i:15+i]) for i in (0,9,18)]\n  except ValueError: box=None\n placeholder=bool(box and min(box)<=2.0)\n models=sum(x.startswith(\'MODEL \') for x in lines)\n atomlines=[x for x in lines if x.startswith((\'ATOM  \',\'HETATM\'))]\n keys=[(x[21:22],x[22:27],x[12:16],x[16:17],x[17:20]) for x in atomlines]\n duplicates=[str(k) for k,n in Counter(keys).items() if n>1]\n altloc=sum(bool(x[16:17].strip()) for x in atomlines)\n parser=PDBParser(QUIET=True,PERMISSIVE=False)\n structure=parser.get_structure(\'input\',str(path)); model=next(structure.get_models())\n chains={c.id:list(c.get_residues()) for c in model}; protein=chains.get(\'A\',[]); rna=chains.get(\'E\',[])\n def one(r): return seq1(r.resname,custom_map={\'UNK\':\'X\'},undef_code=\'X\')\n protseq=\'\'.join(one(r) for r in protein); rnaseq=\'\'.join(r.resname.strip() if r.resname.strip() in RNA else \'?\' for r in rna)\n ids=[r.id[1] for r in rna]\n missing=[]\n for j,r in enumerate(rna):\n  base=r.resname.strip()\n  if base not in BASE: missing.append({\'residue\':r.id[1],\'error\':\'unsupported residue \'+base}); continue\n  need=SUGAR|BASE[base]\n  if j>0: need=need|{\'P\',\'OP1\',\'OP2\'}\n  m=sorted(need-{a.name for a in r})\n  if m: missing.append({\'residue\':r.id[1],\'missing_heavy_atoms\':m})\n for r in protein:\n  m=sorted({\'N\',\'CA\',\'C\',\'O\'}-{a.name for a in r})\n  if m: missing.append({\'chain\':\'A\',\'residue\':r.id[1],\'missing_backbone_atoms\':m})\n def heavy(rs): return [a for r in rs for a in r if a.element not in (\'H\',\'D\') and a.element]\n pa,ra=heavy(protein),heavy(rna); contacts={\'pairs_lt_2A\':None,\'minimum_A\':None,\'closest_pair\':None,\'criterion\':\'all protein–RNA heavy-atom pairs < 2.0 Å; no interchain covalent bonds assumed\'}\n if pa and ra:\n  px=np.array([a.coord for a in pa]); rx=np.array([a.coord for a in ra]); tree=cKDTree(px); close=tree.query_ball_point(rx,2.0)\n  contacts[\'pairs_lt_2A\']=sum(float(np.linalg.norm(rx[j]-px[i]))<2.0 for j,ix in enumerate(close) for i in ix)\n  dist,idx=tree.query(rx); j=int(np.argmin(dist)); a,b=pa[int(idx[j])],ra[j]\n  def atomid(x): return {\'chain\':x.parent.parent.id,\'residue\':x.parent.id[1],\'resname\':x.parent.resname,\'atom\':x.name}\n  contacts[\'minimum_A\']=float(dist[j]);contacts[\'closest_pair\']=[atomid(a),atomid(b)]\n bonds=[]\n for left,right in zip(rna,rna[1:]):\n  d=float(np.linalg.norm(left["O3\'"].coord-right[\'P\'].coord)) if "O3\'" in left and \'P\' in right else None\n  bonds.append({\'from_residue\':left.id[1],\'to_residue\':right.id[1],\'O3_P_A\':d,\'within_1_3_to_2_0_A\':d is not None and 1.3<=d<=2.0})\n peptide=[]\n for left,right in zip(protein,protein[1:]):\n  d=float(np.linalg.norm(left[\'C\'].coord-right[\'N\'].coord)) if \'C\' in left and \'N\' in right else None\n  if d is None or not 1.1<=d<=1.8: peptide.append({\'from\':left.id[1],\'to\':right.id[1],\'C_N_A\':d})\n five_prime_atoms=sorted(a.name for a in rna[0]) if rna else []\n five_prime_phosphate=any(n in five_prime_atoms for n in (\'P\',\'OP1\',\'OP2\',\'OP3\'))\n errors=[]\n if five_prime_phosphate: errors.append(\'5-prime terminal phosphate atoms conflict with the requested 5-prime hydroxyl; explicit chemical conversion is required\')\n if placeholder: errors.append(\'CRYST1 placeholder box must be removed or overridden before periodic simulation\')\n if models>1: errors.append(\'Multiple MODEL records\')\n if duplicates or altloc: errors.append(\'Duplicate atoms or alternate locations require explicit review\')\n if len(protein)!=360 or [r.id[1] for r in protein]!=list(range(1,361)): errors.append(\'Chain A is not the expected residues 1–360\')\n if ids!=list(range(37,68)) or rnaseq!=\'A\'*31: errors.append(\'Chain E is not expected 31 adenines E37–E67\')\n if uniprot_sequence is not None and protseq!=uniprot_sequence[10:370]: errors.append(\'Chain A sequence does not exactly match P11940 residues 11–370\')\n if missing: errors.append(\'Missing expected heavy atoms or unsupported residues\')\n if any(not b[\'within_1_3_to_2_0_A\'] for b in bonds): errors.append(\'RNA linkage outlier or missing bond atoms\')\n if peptide: errors.append(\'Protein peptide connectivity outlier\')\n if contacts[\'minimum_A\'] is not None and contacts[\'minimum_A\']<2: errors.append(\'Protein–RNA contacts < 2 Å require structural review\')\n return {\'input_file\':path.name,\'sha256\':sha256(path),\'model_records\':models,\'chain_residue_counts\':{k:len(v) for k,v in chains.items()},\'protein_sequence\':protseq,\'uniprot_mapping_11_370_match\':None if uniprot_sequence is None else protseq==uniprot_sequence[10:370], \'rna_sequence\':rnaseq,\'rna_mapping\':[{\'position\':j+1,\'chain\':\'E\',\'residue\':r.id[1],\'base\':r.resname} for j,r in enumerate(rna)],\'rna_termini\':{\'five_prime_residue\':rna[0].id[1] if rna else None,\'five_prime_atoms\':five_prime_atoms,\'five_prime_phosphate_present\':five_prime_phosphate,\'intended_five_prime_chemistry\':\'5-prime hydroxyl\',\'intended_three_prime_chemistry\':\'3-prime hydroxyl; verify\'},\'cryst1\':{\'line\':cryst,\'lengths_A\':box,\'placeholder_detected\':placeholder},\'duplicate_atom_records\':duplicates,\'alternate_location_atom_records\':altloc,\'missing_atoms\':missing,\'rna_linkages\':bonds,\'peptide_linkage_outliers\':peptide,\'contacts\':contacts,\'preflight_errors\':errors,\'limitations\':[\'Side-chain completeness, absolute stereochemistry and force-field matching are not fully validated by this audit.\',\'A 5-prime hydroxyl and 3-prime hydroxyl are intended but terminal chemistry needs explicit atom/bond and template verification.\',\'No relaxation or repair was performed.\']}\n\ndef normalized_length_analysis(utr50,utr15):\n if not isinstance(utr50,str) or not isinstance(utr15,str): raise ValueError(\'missing sequence\')\n s=utr50.upper().strip(); short=utr15.upper().strip()\n if len(s)!=50 or len(short)!=15 or s[-15:]!=short or set(s)-RNA: raise ValueError(\'UTR50 does not agree with UTR15\')\n run=len(s)-len(s.rstrip(\'A\')); end=50-run\n if end<16: raise ValueError(\'fewer than 16 observed bases through the last non-A; cannot derive construct\')\n return {\'native_terminal_A_count\':run,\'normalized_UTR16\':s[end-16:end],\'candidate_sequence\':s[end-16:]+\'A\'*15,\'candidate_length\':31+run,\'fixed_31_supported\':run==0}\n')
(TOOLS/'reference_runner.py').write_text('"""Experimental reference-only restrained preparation. No affinity prediction.\n\nGenerates *candidate* structures and detailed quality reports. Geometry and\nconvergence checks are necessary, not sufficient, for biological validation.\n"""\nfrom pathlib import Path\nimport argparse, csv, hashlib, json, math, os, random, shutil, time, urllib.request\nimport numpy as np\nfrom scipy.spatial import cKDTree\nimport openmm as mm\nfrom openmm import app, unit\nimport preflight\nfrom preflight import audit_pdb\n\nRNA_BB={"P","OP1","OP2","O5\'","C5\'","C4\'","O4\'","C3\'","O3\'","C2\'","O2\'","C1\'"}\nFF_NAMES=[\'amber19/protein.ff19SB.xml\',\'amber14/RNA.OL3.xml\',\'amber19/tip3p.xml\']\n\ndef digest(p):return hashlib.sha256(Path(p).read_bytes()).hexdigest()\ndef atomic_json(p,obj):\n p=Path(p);t=p.with_suffix(p.suffix+\'.tmp\');t.write_text(json.dumps(obj,indent=2,allow_nan=False));t.replace(p)\ndef context_for(system,integrator,platform):\n order=[\'CUDA\',\'CPU\'] if platform==\'auto\' else [platform]\n errors=[]\n for name in order:\n  try:\n   p=mm.Platform.getPlatformByName(name);opts={\'Threads\':os.environ.get(\'OPENMM_CPU_THREADS\',\'2\')} if name==\'CPU\' else {}\n   return mm.Context(system,integrator,p,opts),name,errors\n  except Exception as e:errors.append(f\'{name}: {type(e).__name__}: {e}\')\n raise RuntimeError(\'No requested OpenMM platform available: \'+\'; \'.join(errors))\n\ndef selections(topology):\n atoms=list(topology.atoms()); protein=[];rna=[];ca=[];protein_bb=[];bb=[];stereo=[];links=[]\n residues=list(topology.residues())\n for a in atoms:\n  if a.element is None:raise ValueError(f\'Atom has no element: {a}\')\n  if a.residue.chain.id==\'A\':\n   if a.element.symbol not in (\'H\',\'D\'):protein.append(a.index)\n   if a.name==\'CA\':ca.append(a.index)\n   if a.name in (\'N\',\'CA\',\'C\',\'O\'):protein_bb.append(a.index)\n  if a.residue.chain.id==\'E\':\n   if a.element.symbol not in (\'H\',\'D\'):rna.append(a.index)\n   if a.name in RNA_BB:bb.append(a.index)\n rna_res=[r for r in residues if r.chain.id==\'E\'];protein_res=[r for r in residues if r.chain.id==\'A\']\n if len(rna_res)!=31 or len(protein_res)!=360 or len(ca)!=360:raise ValueError(\'Reference chain/residue counts differ from E31/A360\')\n for r in rna_res+protein_res:\n  ns={a.name:a.index for a in r.atoms()};spec=[]\n  if r.chain.id==\'E\':\n   gly=\'N9\' if r.name in (\'A\',\'G\') else \'N1\'\n   spec=[("C1\'",("O4\'","C2\'",gly),1),("C2\'",("C1\'","C3\'","O2\'"),-1),("C3\'",("C2\'","C4\'","O3\'"),-1),("C4\'",("C3\'","O4\'","C5\'"),1)]\n  elif r.name!=\'GLY\':spec=[(\'CA\',(\'N\',\'C\',\'CB\'),1)]\n  if r.chain.id==\'A\' and r.name==\'ILE\':spec.append((\'CB\',(\'CA\',\'CG1\',\'CG2\'),1))\n  if r.chain.id==\'A\' and r.name==\'THR\':spec.append((\'CB\',(\'CA\',\'OG1\',\'CG2\'),1))\n  for center,group,sgn in spec:\n   if not all(k in ns for k in (center,*group)):raise ValueError(f\'Cannot check chirality {r} {center}\')\n   stereo.append({\'residue\':str(r),\'center\':center,\'indices\':[ns[k] for k in (center,*group)],\'expected_sign\':sgn})\n for l,r in zip(rna_res,rna_res[1:]):\n  x={a.name:a.index for a in l.atoms()};y={a.name:a.index for a in r.atoms()};links.append((x["O3\'"],y[\'P\']))\n return {\'protein\':protein,\'rna\':rna,\'protein_ca\':ca,\'protein_backbone\':protein_bb,\'rna_backbone\':bb,\'stereo\':stereo,\'links\':links}\n\ndef rms(a):return float(np.sqrt(np.mean(np.sum(a*a,axis=1))))\ndef diagnostics(x,initial,sel):\n # All positions in nm, converted to Å for geometry.\n xyz=x*10; ref=initial*10;ca=sel[\'protein_ca\'];bb=sel[\'rna_backbone\']\n ref_c=ref[ca].mean(0);mob_c=xyz[ca].mean(0);u,s,vt=np.linalg.svd((xyz[ca]-mob_c).T@(ref[ca]-ref_c));rot=u@np.diag([1,1,np.linalg.det(u@vt)])@vt; aligned=(xyz-mob_c)@rot+ref_c\n rna=sel[\'rna\'];protein=sel[\'protein\'];tree=cKDTree(xyz[protein]);near=tree.query_ball_point(xyz[rna],2.0)\n pair_count=int(sum(np.linalg.norm(xyz[rna[j]]-xyz[protein[i]])<2.0 for j,ns in enumerate(near) for i in ns));minimum=float(tree.query(xyz[rna])[0].min())\n bad=[];volume_min=float(\'inf\')\n for row in sel[\'stereo\']:\n  center,*nb=row[\'indices\'];v=float(np.linalg.det(np.array([xyz[i]-xyz[center] for i in nb])));signed=v*row[\'expected_sign\'];volume_min=min(volume_min,signed)\n  if signed<0.5:bad.append({\'residue\':row[\'residue\'],\'center\':row[\'center\'],\'signed_volume_A3\':v})\n distances=[float(np.linalg.norm(xyz[i]-xyz[j])) for i,j in sel[\'links\']]\n return {\'protein_CA_RMSD_protein_aligned_A\':rms(aligned[ca]-ref[ca]),\'protein_backbone_RMSD_protein_aligned_A\':rms(aligned[sel[\'protein_backbone\']]-ref[sel[\'protein_backbone\']]),\'RNA_backbone_RMSD_protein_aligned_A\':rms(aligned[bb]-ref[bb]),\'protein_CA_max_displacement_unaligned_A\':float(np.linalg.norm(xyz[ca]-ref[ca],axis=1).max()),\'RNA_backbone_max_displacement_unaligned_A\':float(np.linalg.norm(xyz[bb]-ref[bb],axis=1).max()),\'interchain_minimum_A\':minimum,\'interchain_pairs_lt_2A\':pair_count,\'RNA_O3_P_range_A\':[min(distances),max(distances)],\'RNA_bad_link_count\':sum(not 1.3<=x<=2.0 for x in distances),\'checked_chiral_centers\':len(sel[\'stereo\']),\'smallest_expected_signed_volume_A3\':volume_min,\'stereochemistry_outliers\':bad}\n\ndef validate_geometry(d,require_no_interchain_clash):\n issues=[]\n if d[\'stereochemistry_outliers\']:issues.append(\'inverted or nearly planar stereocenters\')\n if d[\'RNA_bad_link_count\']:issues.append(\'RNA phosphodiester distance outliers\')\n if d[\'protein_CA_RMSD_protein_aligned_A\']>1.0 or d[\'RNA_backbone_RMSD_protein_aligned_A\']>1.0:issues.append(\'protein or RNA backbone RMSD exceeds the conservative 1 Å review threshold\')\n if d[\'protein_CA_max_displacement_unaligned_A\']>3.0 or d[\'RNA_backbone_max_displacement_unaligned_A\']>3.0:issues.append(\'restrained backbone atom displacement exceeds the conservative 3 Å review threshold\')\n if require_no_interchain_clash and d[\'interchain_pairs_lt_2A\']:issues.append(\'interchain heavy-atom pairs remain below 2 Å\')\n if not math.isfinite(d[\'interchain_minimum_A\']):issues.append(\'nonfinite geometry\')\n if issues:raise RuntimeError(\'; \'.join(issues))\n\ndef convergence_observed(last,tolerance):\n return bool(last is not None and last[\'objective_gradient_rms_kJ_mol_nm\']<=tolerance and last[\'max_constraint_error\']<=1e-5)\n\nclass Tracker(mm.MinimizationReporter):\n def __init__(self):super().__init__();self.calls=0;self.last=None\n def report(self,iteration,x,grad,args):\n  self.calls+=1;g=np.asarray(grad,dtype=float)\n  self.last={\'iteration\':int(iteration),\'objective_gradient_rms_kJ_mol_nm\':float(np.sqrt(np.mean(g*g))),\'max_constraint_error\':float(args[\'max constraint error\'])}\n  return False\n\ndef stereo_barrier(sel):\n # Numerical anti-inversion aid. It is not part of ff19SB/OL3 and must be\n # reported separately; zero final energy does not make the pathway unbiased.\n expr=(\'0.5*kc*step(vmin-s*v)*(vmin-s*v)^2;\'\n       \'v=(x2-x1)*((y3-y1)*(z4-z1)-(z3-z1)*(y4-y1))\'\n       \'-(y2-y1)*((x3-x1)*(z4-z1)-(z3-z1)*(x4-x1))\'\n       \'+(z2-z1)*((x3-x1)*(y4-y1)-(y3-y1)*(x4-x1))\')\n force=mm.CustomCompoundBondForce(4,expr);force.addGlobalParameter(\'kc\',1e12);force.addGlobalParameter(\'vmin\',.001);force.addPerBondParameter(\'s\');force.setForceGroup(3)\n for row in sel[\'stereo\']:force.addBond(row[\'indices\'],[row[\'expected_sign\']])\n return force\n\ndef restraint_force(name,indices,reference,group):\n f=mm.CustomExternalForce(f\'0.5*{name}*((x-x0)^2+(y-y0)^2+(z-z0)^2)\');f.addGlobalParameter(name,0.0)\n for p in (\'x0\',\'y0\',\'z0\'):f.addPerParticleParameter(p)\n for i in indices:f.addParticle(i,reference[i])\n f.setForceGroup(group);return f\n\ndef save_pdb(path,top,pos):\n with open(path,\'w\') as fh:app.PDBFile.writeFile(top,pos,fh,keepIds=True)\n loaded=app.PDBFile(str(path))\n if loaded.topology.getNumAtoms()!=top.getNumAtoms():raise RuntimeError(f\'Reloaded atom count mismatch in {path}\')\n\ndef minimize_with_retries(ctx,tolerance,iterations):\n # Resetting L-BFGS can help when it exits early. It does not prove convergence.\n remaining=iterations;attempts=[];tracker=None\n for _ in range(3):\n  if remaining<=0:break\n  tracker=Tracker();budget=remaining;mm.LocalEnergyMinimizer.minimize(ctx,tolerance,budget,tracker)\n  attempts.append({\'iteration_budget\':budget,\'reporter_calls\':tracker.calls,\'last_reporter\':tracker.last})\n  if convergence_observed(tracker.last,tolerance) or tracker.calls==0 or tracker.calls>=budget:break\n  remaining-=tracker.calls\n return tracker,attempts\n\ndef minimize_stage(name,ctx,top,sel,original,out,iterations,all_k,anchor_k,tolerance=10.0,soft_stage=False,allow_unconverged=False,upstream_sha=None):\n ctx.setParameter(\'k_all\',all_k);ctx.setParameter(\'k_anchor\',anchor_k); t=time.monotonic()\n tracker,attempts=minimize_with_retries(ctx,tolerance,iterations)\n state=ctx.getState(getPositions=True,getEnergy=True,getForces=True,enforcePeriodicBox=False)\n pos=state.getPositions(asNumpy=True);xyz=np.asarray(pos.value_in_unit(unit.nanometer));force=np.asarray(state.getForces(asNumpy=True).value_in_unit(unit.kilojoule_per_mole/unit.nanometer))\n if not np.isfinite(xyz).all() or not np.isfinite(force).all():raise RuntimeError(\'Nonfinite coordinates or forces in \'+name)\n def energy(groups):return float(ctx.getState(getEnergy=True,groups=groups).getPotentialEnergy().value_in_unit(unit.kilojoule_per_mole))\n qa=diagnostics(xyz,original,sel); qa.update({\'stage\':name,\'energy_interpretation\':\'artificial soft repulsion and bonded terms; not full force field\' if soft_stage else \'full force field\',\'seconds\':time.monotonic()-t,\'max_iterations\':iterations,\'base_system_energy_kJ_mol\':energy({0}),\'physical_forcefield_energy_kJ_mol\':None if soft_stage else energy({0}),\'positional_restraint_energy_kJ_mol\':energy({1,2}),\'stereochemistry_penalty_kJ_mol\':energy({3}),\'total_energy_kJ_mol\':energy({0,1,2,3}),\'maximum_unprojected_force_kJ_mol_nm\':float(np.linalg.norm(force,axis=1).max()),\'reporter_calls\':sum(a[\'reporter_calls\'] for a in attempts),\'minimizer_attempts\':attempts,\'last_reporter\':tracker.last,\'convergence_status\':\'not established; reporter gradient includes constraint penalties and early termination is possible\',\'restraint_definition\':\'E=0.5*k*sum(displacement^2)\',\'k_all_kJ_mol_nm2\':all_k,\'k_anchor_kJ_mol_nm2\':anchor_k})\n observed=convergence_observed(tracker.last,tolerance)\n qa[\'reporter_threshold_observed\']=observed\n # A diagnostic is the commit marker: it is written only after the PDB and\n # exact OpenMM state are saved, reloaded and hashed.\n pdbpath=out/(name+\'.pdb\');statepath=out/(name+\'_state.xml\')\n save_pdb(pdbpath,top,pos)\n tmppath=out/(name+\'_state.xml.tmp\');tmppath.write_text(mm.XmlSerializer.serialize(state));tmppath.replace(statepath)\n qa[\'artifact_sha256\']={\'pdb\':digest(pdbpath),\'state\':digest(statepath)};qa[\'upstream_sha256\']=upstream_sha\n atomic_json(out/(name+\'_diagnostics.json\'),qa)\n validate_geometry(qa,not soft_stage)\n if not soft_stage and abs(qa[\'stereochemistry_penalty_kJ_mol\'])>1e-6:raise RuntimeError(\'Stereochemistry penalty remains active; do not use this stage as an unbiased force-field minimum\')\n if not soft_stage and not observed and not allow_unconverged:raise RuntimeError(f\'{name} did not show the requested gradient/constraint thresholds; inspect diagnostics; early optimizer termination or poor geometry may persist even with more iterations\')\n print(name, \'iterations reported\',tracker.calls,\'min distance Å\',round(qa[\'interchain_minimum_A\'],3),\'RNA bb RMSD Å\',round(qa[\'RNA_backbone_RMSD_protein_aligned_A\'],3),\'seconds\',round(qa[\'seconds\'],1),flush=True)\n return qa,pos\n\ndef advance_stage(name,ctx,top,sel,original,out,iterations,all_k,anchor_k,soft_stage=False,allow_unconverged=False,upstream_sha=None):\n diag=out/(name+\'_diagnostics.json\');state=out/(name+\'_state.xml\');pdb=out/(name+\'.pdb\')\n if diag.exists() and state.exists() and pdb.exists():\n  try:\n   old=json.loads(diag.read_text());h=old[\'artifact_sha256\']\n   intact=h[\'state\']==digest(state) and h[\'pdb\']==digest(pdb) and old.get(\'upstream_sha256\')==upstream_sha\n  except (ValueError,KeyError,OSError):intact=False\n  if intact:\n   validate_geometry(old,not soft_stage)\n   if not soft_stage and abs(old[\'stereochemistry_penalty_kJ_mol\'])>1e-6:raise RuntimeError(f\'{name}: saved stereochemistry penalty is active; review before continuing\')\n   ctx.setState(mm.XmlSerializer.deserialize(state.read_text()))\n   if soft_stage or old[\'reporter_threshold_observed\']:\n    print(\'Reused completed checkpoint:\',name,flush=True)\n    return old,ctx.getState(getPositions=True).getPositions(asNumpy=True)\n   history=out/(name+\'_history.jsonl\')\n   with open(history,\'a\') as fh:fh.write(json.dumps(old,allow_nan=False)+\'\\n\')\n   print(\'Continuing unconverged stage from saved state:\',name,flush=True)\n  else:print(\'Ignoring incomplete checkpoint and restarting this stage:\',name,flush=True)\n return minimize_stage(name,ctx,top,sel,original,out,iterations,all_k,anchor_k,soft_stage=soft_stage,allow_unconverged=allow_unconverged,upstream_sha=upstream_sha)\n\ndef main(source,out,platform=\'auto\',dry_steps=2000,water_steps=2000,stage_steps=2000,seed=9181,allow_unconverged=False):\n if min(dry_steps,water_steps,stage_steps)<=0:raise ValueError(\'Iteration budgets must be positive; unlimited minimization is disabled\')\n source=Path(source);out=Path(out);out.mkdir(parents=True,exist_ok=True)\n identity={\'source_sha256\':digest(source),\'runner_sha256\':digest(__file__),\'preflight_sha256\':digest(preflight.__file__),\'openmm_version\':mm.__version__,\'seed\':seed,\'forcefields\':FF_NAMES,\'experimental_allow_unconverged\':allow_unconverged}\n identity_path=out/\'run_identity.json\';initial_pdb=out/\'initial_protonated_solute.pdb\';initial_npy=out/\'initial_positions_nm.npy\'\n ff=app.ForceField(*FF_NAMES)\n if identity_path.exists():\n  old=json.loads(identity_path.read_text())\n  if old[\'identity\']!=identity:raise ValueError(\'Cannot resume: input, code, seed or OpenMM version differs from the original run\')\n  if old[\'initial_pdb_sha256\']!=digest(initial_pdb) or old[\'initial_positions_sha256\']!=digest(initial_npy):raise ValueError(\'Cannot resume: initial checkpoint checksum mismatch\')\n  loaded=app.PDBFile(str(initial_pdb));original=np.load(initial_npy,allow_pickle=False)\n  if original.shape!=(loaded.topology.getNumAtoms(),3):raise ValueError(\'Cannot resume: initial atom count mismatch\')\n  mod=app.Modeller(loaded.topology,original*unit.nanometer);print(\'Resuming run from verified checkpoints:\',out,flush=True)\n else:\n  if any(out.iterdir()):raise FileExistsError(\'Output folder contains files but no valid run identity; choose a new folder\')\n  with urllib.request.urlopen(\'https://rest.uniprot.org/uniprotkb/P11940.fasta\',timeout=30) as r:fasta=r.read().decode()\n  seq=\'\'.join(fasta.splitlines()[1:]);a=audit_pdb(source,seq);atomic_json(out/\'initial_audit.json\',a)\n  allowed_errors=(\'5-prime terminal phosphate atoms\',\'CRYST1 placeholder box\',\'Protein–RNA contacts < 2\')\n  other_errors=[error for error in a[\'preflight_errors\'] if not error.startswith(allowed_errors)]\n  if other_errors:raise ValueError(\'Input identity or geometry failures: \'+str(other_errors))\n  if a[\'uniprot_mapping_11_370_match\'] is not True or a[\'rna_sequence\']!=\'A\'*31 or len(a[\'rna_mapping\'])!=31 or [r[\'residue\'] for r in a[\'rna_mapping\']]!=list(range(37,68)) or a[\'missing_atoms\'] or a[\'duplicate_atom_records\'] or a[\'alternate_location_atom_records\'] or a[\'peptide_linkage_outliers\'] or any(not z[\'within_1_3_to_2_0_A\'] for z in a[\'rna_linkages\']) or a[\'model_records\']>1:\n   raise ValueError(\'Input fails mandatory identity, missing atom, or backbone checks; inspect initial_audit.json\')\n  lines=source.read_text().splitlines(keepends=True);terminal=[s for s in lines if s.startswith(\'ATOM  \') and s[21:22]==\'E\' and s[22:26].strip()==\'37\' and s[12:16].strip() in {\'P\',\'OP1\',\'OP2\',\'OP3\'}]\n  names={s[12:16].strip() for s in terminal}\n  if names not in (set(),{\'P\',\'OP1\',\'OP2\'}):raise ValueError(\'Unexpected 5-prime terminal phosphate atoms; manual chemistry review required\')\n  prepared=out/\'input_5OH_no_box.pdb\';prepared.write_text(\'\'.join(s for s in lines if not s.startswith(\'CRYST1\') and s not in terminal))\n  atomic_json(out/\'input_conversion.json\',{\'source_sha256\':digest(source),\'intended_5prime\':\'OH\',\'removed_terminal_atoms\':[s.strip() for s in terminal],\'removed_CRYST1_records\':[s.strip() for s in lines if s.startswith(\'CRYST1\')],\'note\':\'explicit chemical conversion per user request; original is not overwritten\'})\n  pdb=app.PDBFile(str(prepared));mod=app.Modeller(pdb.topology,pdb.positions);random.seed(seed)\n  hp=mm.Platform.getPlatformByName(\'CPU\');mod.addHydrogens(ff,pH=7.4,platform=hp)\n  if ff.getUnmatchedResidues(mod.topology):raise ValueError(\'Unmatched force-field residues: \'+str(ff.getUnmatchedResidues(mod.topology)))\n  def heavy_coordinates(top,positions):\n   pos=np.asarray(positions.value_in_unit(unit.nanometer));return {(a.residue.chain.id,a.residue.id,a.name):pos[a.index] for a in top.atoms() if a.element is not None and a.element.symbol not in (\'H\',\'D\')}\n  original_heavy=heavy_coordinates(pdb.topology,pdb.positions);with_hydrogen=heavy_coordinates(mod.topology,mod.positions)\n  if original_heavy.keys()!=with_hydrogen.keys() or any(np.linalg.norm(original_heavy[k]-with_hydrogen[k])>1e-7 for k in original_heavy):raise ValueError(\'Adding hydrogens changed an existing heavy atom or its coordinates\')\n  rna_res=[r for r in mod.topology.residues() if r.chain.id==\'E\'];first={a.name for a in rna_res[0].atoms()};last={a.name for a in rna_res[-1].atoms()}\n  if not {"O5\'","HO5\'"}<=first or \'P\' in first or not {"O3\'","HO3\'"}<=last:raise ValueError(\'Requested 5-prime OH or assumed 3-prime OH could not be confirmed\')\n  original=np.asarray(mod.positions.value_in_unit(unit.nanometer));save_pdb(initial_pdb,mod.topology,mod.positions);np.save(initial_npy,original,allow_pickle=False)\n  atomic_json(identity_path,{\'identity\':identity,\'initial_pdb_sha256\':digest(initial_pdb),\'initial_positions_sha256\':digest(initial_npy)})\n sel=selections(mod.topology);natoms=len(original)\n validate_geometry(diagnostics(original,original,sel),False)\n if (out/\'final_status.json\').exists():\n  done=json.loads((out/\'final_status.json\').read_text());candidate=out/\'reference_candidate_solute.pdb\'\n  if candidate.exists() and done.get(\'candidate_sha256\')==digest(candidate):print(\'Verified final candidate already exists:\',candidate,flush=True);return\n  raise ValueError(\'Final status exists but candidate is missing or changed\')\n platform_info={\'requested\':platform,\'openmm_version\':mm.__version__,\'forcefields\':FF_NAMES,\'seed\':seed,\'salt\':\'0.089 M KCl; a proxy for potassium acetate, without magnesium, glycerol or other assay components\',\'water\':\'TIP3P; rigidWater=True\',\'ions\':\'K+, Cl-\',\'experimental_allow_unconverged\':allow_unconverged,\'terminal_chemistry\':\'5-prime OH; 3-prime OH assumed\',\'pH_for_hydrogen_placement\':7.4,\'padding_nm_per_axis\':1.0,\'caveats\':[\'PABPC1 is a truncated model with artificial charged termini.\',\'Force field TIP3P is a modeling choice; ff19SB was developed with OPC.\',\'Soft repulsion, vacuum preconditioning and the anti-inversion penalty are artificial preparation steps; pathway bias remains even when the penalty is zero at the end.\',\'Reference binding geometry is not experimentally validated by minimization.\']}\n atomic_json(out/\'configuration.json\',platform_info)\n # Reduce catastrophic overlaps using an explicitly nonphysical repulsive ramp.\n # Keep canonical bonded forces; omit electrostatics and replace nonbonded LJ\n # only for this preparation stage. No such energy is a force-field energy.\n softsys=ff.createSystem(mod.topology,nonbondedMethod=app.NoCutoff,constraints=None,removeCMMotion=False)\n nbidx=next(i for i in range(softsys.getNumForces()) if isinstance(softsys.getForce(i),mm.NonbondedForce));nb=softsys.getForce(nbidx)\n exclusions=[tuple(int(x) for x in nb.getExceptionParameters(i)[:2]) for i in range(nb.getNumExceptions())]\n soft=mm.CustomNonbondedForce(\'0.5*krep*step(0.5*(diam1+diam2)-r)*(r-0.5*(diam1+diam2))^2\');soft.addGlobalParameter(\'krep\',1000.);soft.addPerParticleParameter(\'diam\');soft.setNonbondedMethod(mm.CustomNonbondedForce.CutoffNonPeriodic);soft.setCutoffDistance(.30)\n for atom in mod.topology.atoms():soft.addParticle([.17 if atom.element.symbol==\'H\' else .27])\n for i,j in exclusions:soft.addExclusion(i,j)\n softsys.removeForce(nbidx);softsys.addForce(soft);softsys.addForce(restraint_force(\'k_all\',[],original,1));softsys.addForce(restraint_force(\'k_anchor\',sel[\'protein_ca\']+sel[\'rna_backbone\'],original,2));softsys.addForce(stereo_barrier(sel))\n inte=mm.VerletIntegrator(.001);ctx,platform_used,errs=context_for(softsys,inte,platform);ctx.setPositions(mod.positions)\n platform_info.update({\'soft_platform\':platform_used,\'soft_platform_fallback_errors\':errs,\'soft_krep_schedule\':[1000.,10000.,100000.,1000000.], \'soft_steps_per_stage\':300,\'soft_particle_diameters_nm\':{\'H\':.17,\'other\':.27},\'soft_cutoff_nm\':.30,\'soft_exclusions\':\'all standard nonbonded exceptions, including 1-4 terms\',\'backbone_RMSD_review_threshold_A\':1.0,\'max_restrained_atom_displacement_review_threshold_A\':3.0,\'chirality_barrier_k_kJ_mol_nm6\':1e12,\'chirality_barrier_min_volume_nm3\':.001});atomic_json(out/\'configuration.json\',platform_info)\n for i,repulsion in enumerate((1000.,10000.,100000.,1000000.),1):\n  ctx.setParameter(\'krep\',repulsion);upstream=digest(initial_npy if i==1 else out/f\'soft_precondition_{i-1}_state.xml\');d,positions=advance_stage(f\'soft_precondition_{i}\',ctx,mod.topology,sel,original,out,300,0.,4184.,soft_stage=True,upstream_sha=upstream)\n del ctx,inte\n dry=ff.createSystem(mod.topology,nonbondedMethod=app.NoCutoff,constraints=None,removeCMMotion=False)\n dry.addForce(restraint_force(\'k_all\',[],original,1));dry.addForce(restraint_force(\'k_anchor\',sel[\'protein_ca\']+sel[\'rna_backbone\'],original,2));dry.addForce(stereo_barrier(sel))\n inte=mm.VerletIntegrator(.001);ctx,platform_used,errs=context_for(dry,inte,platform);ctx.setPositions(positions)\n platform_info.update({\'dry_platform\':platform_used,\'dry_platform_fallback_errors\':errs});atomic_json(out/\'configuration.json\',platform_info)\n d,positions=advance_stage(\'dry_precondition_exploratory\',ctx,mod.topology,sel,original,out,dry_steps,0.,4184.,allow_unconverged=allow_unconverged,upstream_sha=digest(out/\'soft_precondition_4_state.xml\'));del ctx,inte\n mod.positions=positions\n # Cache the exact solvated starting coordinates and System before stage 0.\n # A marker with hashes is written last so incomplete writes cannot be reused.\n solvent_meta=out/\'solvent_initial_manifest.json\';solvent_pdb=out/\'solvent_initial.pdb\';solvent_npy=out/\'solvent_initial_positions_nm.npy\';system_path=out/\'solvated_system.xml\'\n if solvent_meta.exists():\n  metadata=json.loads(solvent_meta.read_text())\n  if metadata.get(\'dry_state_sha256\')!=digest(out/\'dry_precondition_exploratory_state.xml\'):raise ValueError(\'Cannot resume: solvated checkpoint was built from a different dry state; choose a new output folder\')\n  for key,path in [(\'pdb\',solvent_pdb),(\'positions\',solvent_npy),(\'system\',system_path)]:\n   if not path.exists() or digest(path)!=metadata[\'sha256\'][key]:raise ValueError(\'Cannot resume: solvated \'+key+\' checkpoint checksum mismatch\')\n  saved=app.PDBFile(str(solvent_pdb));sol_x=np.load(solvent_npy,allow_pickle=False)\n  if sol_x.shape!=(saved.topology.getNumAtoms(),3):raise ValueError(\'Cannot resume: solvated atom count mismatch\')\n  def signatures(top):return [(a.residue.chain.id,a.residue.id,a.residue.name,a.name) for a in top.atoms()]\n  if signatures(saved.topology)[:natoms]!=signatures(mod.topology):raise ValueError(\'Cannot resume: solvated solute atom mapping changed\')\n  mod=app.Modeller(saved.topology,sol_x*unit.nanometer);system=mm.XmlSerializer.deserialize(system_path.read_text())\n  print(\'Reused verified solvated system:\',mod.topology.getNumAtoms(),\'atoms\',flush=True)\n else:\n  span=np.ptp(np.asarray(positions.value_in_unit(unit.nanometer)),axis=0);box=mm.Vec3(*(span+2.0))*unit.nanometer\n  random.seed(seed);mod.addSolvent(ff,model=\'tip3p\',boxSize=box,positiveIon=\'K+\',negativeIon=\'Cl-\',ionicStrength=0.089*unit.molar,neutralize=True)\n  system=ff.createSystem(mod.topology,nonbondedMethod=app.PME,nonbondedCutoff=.9*unit.nanometer,constraints=None,rigidWater=True,removeCMMotion=False)\n  sol_x=np.asarray(mod.positions.value_in_unit(unit.nanometer));system.addForce(restraint_force(\'k_all\',list(range(natoms)),sol_x,1));system.addForce(restraint_force(\'k_anchor\',sel[\'protein_ca\']+sel[\'rna_backbone\'],original,2));system.addForce(stereo_barrier(sel))\n  save_pdb(solvent_pdb,mod.topology,mod.positions);np.save(solvent_npy,sol_x,allow_pickle=False);system_path.write_text(mm.XmlSerializer.serialize(system))\n  atomic_json(solvent_meta,{\'dry_state_sha256\':digest(out/\'dry_precondition_exploratory_state.xml\'),\'sha256\':{\'pdb\':digest(solvent_pdb),\'positions\':digest(solvent_npy),\'system\':digest(system_path)},\'total_atoms\':mod.topology.getNumAtoms(),\'solute_atoms\':natoms})\n inte=mm.VerletIntegrator(.001);ctx,platform_used,errs=context_for(system,inte,platform);ctx.setPositions(mod.positions)\n vectors=system.getDefaultPeriodicBoxVectors();lengths=[float(v[i].value_in_unit(unit.nanometer)) for i,v in enumerate(vectors)]\n platform_info.update({\'solvated_platform\':platform_used,\'solvated_platform_fallback_errors\':errs,\'solute_atoms\':natoms,\'total_atoms\':mod.topology.getNumAtoms(),\'box_lengths_nm\':lengths});atomic_json(out/\'configuration.json\',platform_info)\n stages=[(\'stage0_water\',water_steps,418400.,0.),(\'stage1_strong\',stage_steps,0.,4184.),(\'stage2_moderate\',stage_steps,0.,1255.2)]\n results=[]\n for idx,(name,iters,kall,kanchor) in enumerate(stages):\n  upstream=digest(solvent_npy if idx==0 else out/(stages[idx-1][0]+\'_state.xml\'))\n  d,positions=advance_stage(name,ctx,mod.topology,sel,original,out,iters,kall,kanchor,allow_unconverged=allow_unconverged,upstream_sha=upstream);results.append(d)\n mod.positions=positions\n # Make a separate solute-only file; the full solvated system is retained above.\n mod.delete([a for a in mod.topology.atoms() if a.index>=natoms]);save_pdb(out/\'reference_candidate_solute.pdb\',mod.topology,mod.positions)\n atomic_json(out/\'final_status.json\',{\'status\':\'candidate_for_review_not_scientifically_validated\',\'candidate_sha256\':digest(out/\'reference_candidate_solute.pdb\'),\'all_requested_stages_ran\':True,\'reporter_thresholds_observed_all_solvated\':all(r[\'reporter_threshold_observed\'] for r in results),\'convergence_proven\':False,\'notes\':[\'Gradient and constraint details are in stage diagnostics; iteration caps do not prove convergence.\',\'No claim of biological pose validation or binding affinity.\',\'Within-stage interruption replays that stage from its last saved checkpoint; platform changes may alter numerical results.\',\'All-atom clash and independent stereochemical review remain required.\'],\'stages\':results})\n print(\'Candidate completed; independent review and convergence assessment remain required.\',flush=True)\nif __name__==\'__main__\':\n p=argparse.ArgumentParser();p.add_argument(\'source\');p.add_argument(\'out\');p.add_argument(\'--platform\',choices=[\'auto\',\'CPU\',\'CUDA\',\'Reference\'],default=\'auto\');p.add_argument(\'--dry-steps\',type=int,default=2000);p.add_argument(\'--water-steps\',type=int,default=2000);p.add_argument(\'--stage-steps\',type=int,default=2000);p.add_argument(\'--seed\',type=int,default=9181);p.add_argument(\'--experimental-allow-unconverged\',action=\'store_true\');a=p.parse_args();main(a.source,a.out,a.platform,a.dry_steps,a.water_steps,a.stage_steps,a.seed,a.experimental_allow_unconverged)\n')
(TOOLS/'mutation_smoke_runner.py').write_text('"""Test independent ChimeraX A->G/C/U substitutions; not a batch pipeline."""\nfrom pathlib import Path\nimport json,os,random,shutil,subprocess,sys\nimport numpy as np\nfrom Bio.PDB import PDBParser\nimport openmm as mm\nfrom openmm import app,unit\nfrom reference_runner import FF_NAMES,RNA_BB,selections,diagnostics,validate_geometry\nfrom preflight import BASE\n\ndef run(reference,destination,exe=None):\n reference=Path(reference).resolve();out=Path(destination).resolve();out.mkdir(parents=True,exist_ok=True)\n if any(out.iterdir()):raise FileExistsError(\'Mutation test folder must be empty\')\n shutil.copyfile(reference,out/\'reference.pdb\')\n scripts={}\n for target in (\'G\',\'C\',\'U\'):\n  f=out/f\'A_to_{target}.cxc\';f.write_text(f\'open reference.pdb\\nswapna /E:40 {target} preserve true\\nsave A_to_{target}_raw.pdb models #1\\nexit\\n\');scripts[target]=f\n if not exe:\n  (out/\'status.json\').write_text(json.dumps({\'status\':\'scripts_generated_only\',\'note\':\'ChimeraX was not run; run each .cxc from this directory using ChimeraX 1.12 and then validate outputs.\'},indent=2))\n  print(\'ChimeraX unavailable: generated scripts only, no structures created.\',flush=True);return\n version=subprocess.run([str(exe),\'--nogui\',\'--version\'],capture_output=True,text=True)\n if version.returncode or \'version: 1.12\' not in version.stdout+version.stderr:raise RuntimeError(\'This smoke test was checked only with ChimeraX 1.12: \'+version.stdout+version.stderr)\n P=PDBParser(QUIET=True);ref=P.get_structure(\'reference\',str(reference));ref_protein=[r.resname for r in ref[0][\'A\']]\n def heavy(s,ch):return {(r.id[1],a.name):a.coord for r in s[0][ch] for a in r if a.element not in (\'H\',\'D\')}\n ref_p=heavy(ref,\'A\');ref_r=heavy(ref,\'E\');ff=app.ForceField(*FF_NAMES);results=[]\n for target in (\'G\',\'C\',\'U\'):\n  try:\n   result=subprocess.run([str(exe),\'--nogui\',\'--exit\',str(scripts[target])],cwd=out,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True)\n   (out/f\'A_to_{target}_chimerax.log\').write_text(result.stdout)\n   p=out/f\'A_to_{target}_raw.pdb\'\n   if result.returncode or not p.exists():raise RuntimeError(\'ChimeraX failed; inspect log\')\n   mutant=P.get_structure(target,str(p));rna=mutant[0][\'E\'];res=rna[40]\n   if [r.resname for r in mutant[0][\'A\']]!=ref_protein:raise ValueError(\'protein sequence changed\')\n   if \'\'.join(r.resname for r in rna)!=\'AAA\'+target+\'A\'*27:raise ValueError(\'wrong RNA sequence\')\n   sugar=RNA_BB;expected=sugar|BASE[target];got={a.name for a in res if a.element not in (\'H\',\'D\')}\n   if got!=expected:raise ValueError(\'base or backbone heavy atoms do not match expected set\')\n   p_atoms=heavy(mutant,\'A\');r_atoms=heavy(mutant,\'E\');keys=[k for k in ref_r if k[1] in sugar]\n   if p_atoms.keys()!=ref_p.keys() or any(k not in r_atoms for k in keys):raise ValueError(\'protein or RNA backbone atom missing\')\n   protein_max=max(float(np.linalg.norm(ref_p[k]-p_atoms[k])) for k in ref_p);backbone_max=max(float(np.linalg.norm(ref_r[k]-r_atoms[k])) for k in keys)\n   if protein_max>.002 or backbone_max>.002:raise ValueError(\'protein or RNA backbone coordinates changed\')\n   bond=\'N9\' if target==\'G\' else \'N1\';bond_length=float(np.linalg.norm(res["C1\'"].coord-res[bond].coord))\n   if not 1.3<=bond_length<=1.6:raise ValueError(\'bad glycosidic attachment distance\')\n   pdb=app.PDBFile(str(p));mod=app.Modeller(pdb.topology,pdb.positions);random.seed(391);mod.addHydrogens(ff,pH=7.4,platform=mm.Platform.getPlatformByName(\'CPU\'))\n   unmatched=ff.getUnmatchedResidues(mod.topology)\n   if unmatched:raise ValueError(\'unmatched residues: \'+str(unmatched))\n   def heavy_coords(top,positions):\n    v=np.asarray(positions.value_in_unit(unit.nanometer));return {(a.residue.chain.id,a.residue.id,a.name):v[a.index] for a in top.atoms() if a.element and a.element.symbol not in (\'H\',\'D\')}\n   h0=heavy_coords(pdb.topology,pdb.positions);h1=heavy_coords(mod.topology,mod.positions)\n   if h0.keys()!=h1.keys() or any(np.linalg.norm(h0[k]-h1[k])>1e-7 for k in h0):raise ValueError(\'adding hydrogens changed a pre-existing heavy atom\')\n   sel=selections(mod.topology);xyz=np.asarray(mod.positions.value_in_unit(unit.nanometer));d0=diagnostics(xyz,xyz,sel);validate_geometry(d0,False)\n   system=ff.createSystem(mod.topology,nonbondedMethod=app.NoCutoff,constraints=None,removeCMMotion=False)\n   force=mm.CustomExternalForce(\'0.5*k*((x-x0)^2+(y-y0)^2+(z-z0)^2)\');force.addGlobalParameter(\'k\',4184.)\n   for k in (\'x0\',\'y0\',\'z0\'):force.addPerParticleParameter(k)\n   for i in sel[\'protein_ca\']+sel[\'rna_backbone\']:force.addParticle(i,xyz[i])\n   system.addForce(force);inte=mm.VerletIntegrator(.001);ctx=mm.Context(system,inte,mm.Platform.getPlatformByName(\'CPU\'),{\'Threads\':\'2\'});ctx.setPositions(mod.positions)\n   before=ctx.getState(getEnergy=True).getPotentialEnergy().value_in_unit(unit.kilojoule_per_mole);mm.LocalEnergyMinimizer.minimize(ctx,10,25)\n   st=ctx.getState(getPositions=True,getEnergy=True);post=np.asarray(st.getPositions(asNumpy=True).value_in_unit(unit.nanometer));after=st.getPotentialEnergy().value_in_unit(unit.kilojoule_per_mole)\n   if not np.isfinite(post).all() or not np.isfinite(after):raise ValueError(\'nonfinite test results\')\n   d=diagnostics(post,xyz,sel);validate_geometry(d,False)\n   result={\'target\':target,\'status\':\'smoke_test_passed\',\'protein_heavy_max_displacement_A\':protein_max,\'rna_backbone_max_displacement_A\':backbone_max,\'glycosidic_bond_A\':bond_length,\'unmatched_residues\':0,\'post_test_stereo_outliers\':len(d[\'stereochemistry_outliers\']),\'dry_energy_before_kJ_mol\':float(before),\'dry_energy_after_kJ_mol\':float(after),\'iterations_cap\':25,\'limitation\':\'short vacuum smoke test; neither converged nor a variant for scoring\'}\n   del ctx,inte\n  except Exception as e:result={\'target\':target,\'status\':\'failed\',\'error\':str(e)}\n  results.append(result);print(json.dumps(result),flush=True)\n  (out/\'status.json\').write_text(json.dumps({\'status\':\'in_progress\',\'results\':results},indent=2))\n status=\'smoke_tests_finished\' if all(r[\'status\']==\'smoke_test_passed\' for r in results) else \'smoke_tests_failed\'\n (out/\'status.json\').write_text(json.dumps({\'status\':status,\'results\':results},indent=2))\n if status!=\'smoke_tests_finished\':raise RuntimeError(\'One or more smoke tests failed; inspect status.json\')\nif __name__==\'__main__\':run(*sys.argv[1:])\n')
(TOOLS/'tests/test_reference_runner.py').write_text("import sys,unittest\nfrom pathlib import Path\nimport numpy as np\nimport openmm as mm\nfrom openmm import unit\nsys.path.insert(0,str(Path(__file__).resolve().parents[1]))\nfrom reference_runner import Tracker,stereo_barrier,validate_geometry,convergence_observed,minimize_with_retries\n\nclass ReferenceRunnerTests(unittest.TestCase):\n def test_stereo_barrier_is_zero_on_correct_side_and_positive_on_inversion(self):\n  system=mm.System()\n  for i in range(4):system.addParticle(12)\n  system.addForce(stereo_barrier({'stereo':[{'indices':[0,1,2,3],'expected_sign':1}]}))\n  ctx=mm.Context(system,mm.VerletIntegrator(.001),mm.Platform.getPlatformByName('Reference'))\n  ctx.setPositions([[0,0,0],[.14,0,0],[0,.14,0],[0,0,.14]])\n  good=ctx.getState(getEnergy=True).getPotentialEnergy().value_in_unit(unit.kilojoule_per_mole)\n  ctx.setPositions([[0,0,0],[.14,0,0],[0,.14,0],[0,0,-.14]])\n  bad=ctx.getState(getEnergy=True).getPotentialEnergy().value_in_unit(unit.kilojoule_per_mole)\n  self.assertEqual(good,0);self.assertGreater(bad,1e5)\n def test_openmm_reporter_records_real_iterations(self):\n  system=mm.System();system.addParticle(1);f=mm.CustomExternalForce('x*x');f.addParticle(0,[]);system.addForce(f)\n  ctx=mm.Context(system,mm.VerletIntegrator(.001),mm.Platform.getPlatformByName('Reference'));ctx.setPositions([[1,0,0]])\n  t=Tracker();mm.LocalEnergyMinimizer.minimize(ctx,1e-9,5,t)\n  self.assertGreater(t.calls,0);self.assertIn('objective_gradient_rms_kJ_mol_nm',t.last)\n def test_minimizer_retry_wrapper_records_attempts(self):\n  system=mm.System();system.addParticle(1);force=mm.CustomExternalForce('x*x');force.addParticle(0,[]);system.addForce(force)\n  ctx=mm.Context(system,mm.VerletIntegrator(.001),mm.Platform.getPlatformByName('Reference'));ctx.setPositions([[1,0,0]])\n  tracker,attempts=minimize_with_retries(ctx,1e-9,10)\n  self.assertGreaterEqual(len(attempts),1);self.assertGreater(tracker.calls,0)\n def test_reporter_threshold_requires_both_gradient_and_constraint(self):\n  self.assertFalse(convergence_observed(None,10))\n  self.assertTrue(convergence_observed({'objective_gradient_rms_kJ_mol_nm':9,'max_constraint_error':1e-6},10))\n  self.assertFalse(convergence_observed({'objective_gradient_rms_kJ_mol_nm':11,'max_constraint_error':1e-6},10))\n  self.assertFalse(convergence_observed({'objective_gradient_rms_kJ_mol_nm':9,'max_constraint_error':1e-4},10))\n def test_quality_guard_rejects_bad_structures(self):\n  d={'stereochemistry_outliers':[],'RNA_bad_link_count':0,'interchain_pairs_lt_2A':0,'interchain_minimum_A':2.5,'protein_CA_RMSD_protein_aligned_A':.3,'RNA_backbone_RMSD_protein_aligned_A':.2,'protein_CA_max_displacement_unaligned_A':.7,'RNA_backbone_max_displacement_unaligned_A':.8}\n  validate_geometry(d,True)\n  for key,value in [('stereochemistry_outliers',[{'id':'test'}]),('RNA_bad_link_count',1),('interchain_pairs_lt_2A',1),('interchain_minimum_A',float('nan')),('protein_CA_RMSD_protein_aligned_A',1.1),('RNA_backbone_max_displacement_unaligned_A',3.1)]:\n   bad=d.copy();bad[key]=value\n   with self.assertRaises(RuntimeError):validate_geometry(bad,True)\nif __name__=='__main__':unittest.main()\n")
(TOOLS/'tests/test_checkpoints.py').write_text("import sys,unittest,tempfile,json\nfrom unittest.mock import patch\nfrom pathlib import Path\nimport openmm as mm\nsys.path.insert(0,str(Path(__file__).resolve().parents[1]))\nimport reference_runner as rr\n\nclass CheckpointTests(unittest.TestCase):\n def fixture(self,d):\n  s=mm.System();s.addParticle(12);inte=mm.VerletIntegrator(.001);ctx=mm.Context(s,inte,mm.Platform.getPlatformByName('Reference'));ctx.setPositions([[.1,0,0]])\n  state=Path(d)/'s_state.xml';state.write_text(mm.XmlSerializer.serialize(ctx.getState(getPositions=True)))\n  pdb=Path(d)/'s.pdb';pdb.write_text('example atom-file payload')\n  qa={'stereochemistry_outliers':[],'RNA_bad_link_count':0,'interchain_pairs_lt_2A':0,'interchain_minimum_A':2.5,'protein_CA_RMSD_protein_aligned_A':.3,'RNA_backbone_RMSD_protein_aligned_A':.2,'protein_CA_max_displacement_unaligned_A':.7,'RNA_backbone_max_displacement_unaligned_A':.8,'stereochemistry_penalty_kJ_mol':0.0,'reporter_threshold_observed':True,'artifact_sha256':{'state':rr.digest(state),'pdb':rr.digest(pdb)},'upstream_sha256':'expected'}\n  (Path(d)/'s_diagnostics.json').write_text(json.dumps(qa));return ctx,inte,qa\n def test_verified_checkpoint_reused(self):\n  with tempfile.TemporaryDirectory() as d:\n   ctx,inte,qa=self.fixture(d)\n   with patch.object(rr,'minimize_stage',side_effect=AssertionError('unexpected rerun')):\n    result,pos=rr.advance_stage('s',ctx,None,None,None,Path(d),10,0,0,upstream_sha='expected')\n   self.assertEqual(result['upstream_sha256'],'expected')\n def test_changed_upstream_invalidates_checkpoint(self):\n  with tempfile.TemporaryDirectory() as d:\n   ctx,inte,qa=self.fixture(d)\n   with patch.object(rr,'minimize_stage',return_value=('rerun',None)) as mocked:\n    result,_=rr.advance_stage('s',ctx,None,None,None,Path(d),10,0,0,upstream_sha='different')\n   self.assertEqual(result,'rerun');self.assertTrue(mocked.called)\n def test_changed_state_invalidates_checkpoint(self):\n  with tempfile.TemporaryDirectory() as d:\n   ctx,inte,qa=self.fixture(d);(Path(d)/'s.pdb').write_text('tampered')\n   with patch.object(rr,'minimize_stage',return_value=('rerun',None)) as mocked:\n    result,_=rr.advance_stage('s',ctx,None,None,None,Path(d),10,0,0,upstream_sha='expected')\n   self.assertEqual(result,'rerun');self.assertTrue(mocked.called)\nif __name__=='__main__':unittest.main()\n")
if str(TOOLS) not in sys.path: sys.path.insert(0,str(TOOLS))
print('Embedded source unpacked to',TOOLS)


## 3. Test the checks and numerical restraint

In [ ]:
import subprocess,sys
subprocess.run([sys.executable,'-m','unittest','discover','-s',str(TOOLS/'tests'),'-v'],check=True)


## 4. Connect Drive and upload the revised PDB
Drive retains outputs after runtime disconnection. Upload one PDB; the code will verify its sequence and chain mapping before attempting minimization.

In [ ]:
from google.colab import drive, files
from pathlib import Path
import re

drive.mount('/content/drive')
CASE_NAME = 'pose_01'  # Keep this label to resume; use a different label for another pose.
if not re.fullmatch(r'[A-Za-z0-9_-]+',CASE_NAME): raise ValueError('CASE_NAME must contain only letters, digits, underscores or hyphens')
PROJECT=Path('/content/drive/MyDrive/PABPC1_reference_prototype')/CASE_NAME
PROJECT.mkdir(parents=True,exist_ok=True)
INPUT_PDB=PROJECT/'input_original.pdb'
if INPUT_PDB.exists():
    print('Using saved input for this case:',INPUT_PDB)
else:
    uploaded=files.upload();pdbs=[name for name in uploaded if name.lower().endswith('.pdb')]
    if len(pdbs)!=1: raise ValueError('Upload exactly one reference PDB')
    INPUT_PDB.write_bytes(uploaded[pdbs[0]])
print('Output folder:',PROJECT)


## 5. Inspect the GPU and set the iteration budgets

`auto` tries CUDA and falls back to CPU. A GPU assignment is not guaranteed. Each stage has a maximum iteration budget, not a promise of convergence; up to three optimizer attempts may occur if an optimizer exits early. if its check fails, the run stops with diagnostics. Re-run the notebook with the same case label and saved input to resume from verified checkpoints. Do not run the same case in two runtimes simultaneously. A changed input, code, seed or OpenMM version will stop the resume.

In [ ]:
import subprocess
p=subprocess.run(['bash','-lc','nvidia-smi -L'],capture_output=True,text=True)
print(p.stdout if p.returncode==0 else 'No NVIDIA GPU detected; CPU fallback can be much slower.')
PLATFORM = 'auto'  # auto, CUDA, CPU, or Reference
DRY_MAX_ITERATIONS = 2000
WATER_MAX_ITERATIONS = 2000
RESTRAINED_MAX_ITERATIONS = 2000
SEED = 9181


## 6. Run the reference prototype

The endpoint conversion removes the three E37 phosphate atoms because a 5′-OH was explicitly requested; it is logged. The 3′ end is modeled as OH. OpenMM chooses protonation states at pH 7.4. Charged protein termini are artificial fragment termini.

The solvent is TIP3P with approximately 0.089 M KCl as a **simplified proxy** for the 89 mM potassium acetate used by Müller; it omits magnesium, glycerol, and the other assay components. It does not reproduce the full experiment. ff19SB was developed in an OPC-water context; TIP3P is a modeling choice here.

The physical restraint convention is `E=0.5*k*sum(r-r0)^2`: 10 kcal/mol/Å² is 4184 kJ/mol/nm². Full-force-field, positional-restraint, and anti-inversion energies are tracked separately.

In [ ]:
import os, subprocess, sys
RUN_DIR=PROJECT/'reference'
env=os.environ.copy();env['PYTHONPATH']=str(TOOLS);env['OPENMM_CPU_THREADS']='2'
cmd=[sys.executable,'-u',str(TOOLS/'reference_runner.py'),str(INPUT_PDB),str(RUN_DIR),
     '--platform',PLATFORM,'--dry-steps',str(DRY_MAX_ITERATIONS),'--water-steps',str(WATER_MAX_ITERATIONS),
     '--stage-steps',str(RESTRAINED_MAX_ITERATIONS),'--seed',str(SEED)]
print('Running. Progress and artifacts are saved to:',PROJECT)
with open(PROJECT/'run.log','a') as log:
    log.write('\n--- New run or resume ---\n');log.flush()
    proc=subprocess.Popen(cmd,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,env=env,bufsize=1)
    for line in proc.stdout:
        print(line,end='');log.write(line);log.flush()
    exit_code=proc.wait()
if exit_code: raise RuntimeError(f'Reference run stopped with status {exit_code}. Inspect run.log and stage diagnostics. No successful final structure is implied.')
print('Stages finished; read reference/final_status.json and independently review the candidate.')


## 7. Optional: install UCSF ChimeraX 1.12

The user has authorized acceptance of the [UCSF noncommercial license](https://www.rbvi.ucsf.edu/chimerax/docs/licensing.html). Set `INSTALL_CHIMERAX=True` and run this cell to accept it during download and install the official package in this temporary Colab runtime. This downloads approximately 440 MiB and can take several minutes. **The package installation has not been tested on every Colab image.** If installation fails, the next cell can generate scripts without claiming mutations were performed.

In [ ]:
INSTALL_CHIMERAX = False  # Change to True only if you want the optional ChimeraX installation.
import os,shutil,subprocess,urllib.request,urllib.parse,hashlib
from html.parser import HTMLParser
from pathlib import Path

if INSTALL_CHIMERAX and not shutil.which('chimerax'):
    if os.geteuid()!=0: raise RuntimeError('Installing ChimeraX requires the usual root Colab runtime.')
    release={}
    for line in Path('/etc/os-release').read_text().splitlines():
        if '=' in line:
            k,v=line.split('=',1);release[k]=v.strip('"')
    version=release.get('VERSION_ID')
    builds={
      '22.04':('1.12/ubuntu-22.04/ucsf-chimerax_1.12ubuntu22.04_amd64.deb','4044e5fea80e081045ca600f94640b064bd8871352e54bfc05361faa734e292b'),
      '24.04':('1.12/ubuntu-24.04/ucsf-chimerax_1.12ubuntu24.04_amd64.deb','851d6f8933b9587a26e70cc95937acc750e07250a0227ba4af1bfbe972db30ab')}
    if release.get('ID')!='ubuntu' or version not in builds: raise RuntimeError('Unsupported runtime for this installer; install ChimeraX 1.12 by official instructions.')
    name,expected=builds[version]
    service='https://www.rbvi.ucsf.edu/chimerax/cgi-bin/secure/chimerax-get.py'
    payload=urllib.parse.urlencode({'file':name,'choice':'Accept'}).encode()
    with urllib.request.urlopen(urllib.request.Request(service,data=payload),timeout=60) as response: page=response.read().decode()
    class Link(HTMLParser):
        def __init__(self): super().__init__();self.target=None
        def handle_starttag(self,tag,attrs):
            if tag=='a' and 'chimerax-get.py?ident=' in dict(attrs).get('href',''): self.target=dict(attrs)['href']
    parser=Link();parser.feed(page)
    if not parser.target: raise RuntimeError('UCSF download response did not include the expected download link.')
    package=Path('/content/ucsf_chimerax_1.12.deb');print('Downloading official ChimeraX package...')
    with urllib.request.urlopen(urllib.parse.urljoin('https://www.rbvi.ucsf.edu',parser.target),timeout=120) as response, package.open('wb') as fh:
        shutil.copyfileobj(response,fh,length=2**20)
    h=hashlib.sha256()
    with package.open('rb') as fh:
        for block in iter(lambda:fh.read(2**20),b''): h.update(block)
    if h.hexdigest()!=expected: raise RuntimeError('ChimeraX package checksum mismatch; installation stopped.')
    env=os.environ.copy();env['DEBIAN_FRONTEND']='noninteractive'
    subprocess.run(['apt-get','update','-qq'],check=True,env=env)
    subprocess.run(['apt-get','install','-y','--no-install-recommends',str(package)],check=True,env=env)
    package.unlink()
print('ChimeraX executable:',shutil.which('chimerax') or 'not installed')


## 8. Optional canonical mutation smoke test

This uses a fresh copy of the reference for each A→G, A→C and A→U test at E40. ChimeraX changes the base; the checks examine the resulting atom sets, glycosidic bond, protein and RNA-backbone coordinates, force-field assignment and stereochemistry after a **25-iteration dry test**. The mutants are not converged or simulation-ready. If ChimeraX is unavailable, only command scripts are written.

In [ ]:
import os,shutil,subprocess,sys
candidate=RUN_DIR/'reference_candidate_solute.pdb'
if not candidate.exists(): raise FileNotFoundError('A reference candidate was not produced. Resolve the reference diagnostics first.')
smoke_dir=PROJECT/'mutation_smoke'
exe=shutil.which('chimerax')
cmd=[sys.executable,'-u',str(TOOLS/'mutation_smoke_runner.py'),str(candidate),str(smoke_dir)]
if exe: cmd.append(exe)
env=os.environ.copy();env['PYTHONPATH']=str(TOOLS);env['OPENMM_CPU_THREADS']='2'
subprocess.run(cmd,check=True,env=env)
print('See',smoke_dir/'status.json')


## 9. Review before any subsequent calculation

- Open `reference/final_status.json` and every stage diagnostic. The notebook deliberately never claims full scientific validation or exact convergence.
- Inspect all-atom sterics, glycosidic geometry, sugar pucker, protein side-chain rotamers, stereochemistry, RNA connectivity and PABPC1 domain movements with independent tools and expert review. The `<2 Å` check is not a complete clash analysis.
- Do not compare minimized absolute energies across RNA sequences as binding energies.
- This prototype does not yet create the 61 variants, extend RNA beyond 31 nt, process modifications, or guarantee convergence for every input. Those remain separate development tasks.
- References: [OpenMM force fields](https://docs.openmm.org/latest/userguide/application/02_running_sims.html), [ChimeraX swapna](https://www.rbvi.ucsf.edu/chimerax/docs/user/commands/swapna.html), [UniProt P11940](https://www.uniprot.org/uniprotkb/P11940/entry).